## Generate TEEHR crosswalk data from NWM routelink files for all domains

In [ ]:
import xarray as xr
import pandas as pd
from pathlib import Path
import geopandas as gpd
import pandas as pd
from datetime import datetime

import teehr

## Routelink Dates and Info:

#### CONUS:

Current version: https://www.nco.ncep.noaa.gov/pmb/codes/nwprod/nwm.v3.1.6/parm/domain/RouteLink_CONUS.nc
- Read from S3: s3://ciroh-rti-public-data/teehr-data-warehouse/common/crosswalks/routelinks/v3.1/RouteLink_CONUS.nc   
Created 4/12/2019 by NCAR, contains 2776734 feature IDs  
Corresponds to NWM 3.1


#### Hawaii:

Current version: https://www.nco.ncep.noaa.gov/pmb/codes/nwprod/nwm.v3.1.6/parm/domain_hawaii/RouteLink_HI.nc   
Read from S3: s3://ciroh-rti-public-data/teehr-data-warehouse/common/crosswalks/routelinks/v3.1/RouteLink_HI.nc   
Contains 13637 feature IDs   
Corresponds to NWM 3.1  (Hawaii was added in v2.0)   

#### Puerto Rico:

Current version: https://www.nco.ncep.noaa.gov/pmb/codes/nwprod/nwm.v3.1.6/parm/domain_puertorico/RouteLink_PRVI.nc   
Read from S3: s3://ciroh-rti-public-data/teehr-data-warehouse/common/crosswalks/routelinks/v3.1/RouteLink_PRVI.nc   
Contains 14017 feature IDs   
Corresponds to NWM 3.1  (Puerto Rico was added in v2.1)   

#### Alaska:

Current version: https://www.nco.ncep.noaa.gov/pmb/codes/nwprod/nwm.v3.1.6/parm/domain_alaska/RouteLink_AK.nc 
- Read from S3: s3://ciroh-rti-public-data/teehr-data-warehouse/common/crosswalks/routelinks/v3.1/RouteLink_AK.nc   
Created 4/21/2022 by NCAR, contains 391528 feature IDs   
Corresponds to NWM 3.1  (Alaska was added in v3.0)   

#### NWM version dates for reference:
- **v1.2**----------2018-09-17-00 – 2019-06-19-13
- **v2.0**----------2019-06-19-14 – 2021-04-20-13
- **v2.1/v2.2**----2021-04-20-14 – 2023-09-19-11
- **v3.0**----------2023-09-19-12 – 2026-08-17-23
- **v3.1**----------2026-08-18-00 - present

In [ ]:
fetch_date = datetime.now().strftime("%m-%d-%Y")
def extract_nwm_crosswalk_from_routelink(routelink_file, nwm_prefix):

    print(f"Opening file from: {routelink_file["local_path"]}")

    # Open the NetCDF file
    ds = xr.open_dataset(routelink_file["local_path"], engine="h5netcdf", chunks={})

    # extract the links and gages
    df = ds[['link', 'gages']].to_dataframe().reset_index(drop=True)[['link','gages']]

    # remove blanks
    df['gages'] = df['gages'].str.decode('utf-8').str.strip()
    crosswalk = df[df['gages'] != ""].copy().reset_index(drop=True)

    # format into a teehr crosswalk table
    crosswalk['gages'] = crosswalk['gages'].str.zfill(8).radd('usgs-')
    crosswalk['link'] = crosswalk['link'].astype(str).radd(nwm_prefix + '-')
    crosswalk = crosswalk.rename(columns={'link': 'secondary_location_id', 'gages': 'primary_location_id'})
    properties_dict = {
        "source" : "NWS",
        "url" : routelink_file["s3_path"],
        "date_retreived" : fetch_date,
    }
    crosswalk['properties'] = [properties_dict] * len(crosswalk)

    return crosswalk

### Read routelink files and convert to TEEHR format

In [ ]:
%%time
SECONDARY_LOCATION_ID_PREFIX = "nwm31"

routelink_files = {
    "conus_31":{
        "local_path": "/data/common/crosswalks/routelinks/v3.1/RouteLink_CONUS.nc",
        "s3_path": "s3://ciroh-rti-public-data/teehr-data-warehouse/common/crosswalks/routelinks/v3.1/RouteLink_CONUS.nc"
    },
    "hawaii":{
        "local_path": "/data/common/crosswalks/routelinks/v3.1/RouteLink_HI.nc",
        "s3_path": "s3://ciroh-rti-public-data/teehr-data-warehouse/common/crosswalks/routelinks/v3.1/RouteLink_HI.nc"
    },
    "puertorico":{
        "local_path": "/data/common/crosswalks/routelinks/v3.1/RouteLink_PRVI.nc",
        "s3_path": "s3://ciroh-rti-public-data/teehr-data-warehouse/common/crosswalks/routelinks/v3.1/RouteLink_PRVI.nc"
    },
    "alaska":{
        "local_path": "/data/common/crosswalks/routelinks/v3.1/RouteLink_AK.nc",
        "s3_path": "s3://ciroh-rti-public-data/teehr-data-warehouse/common/crosswalks/routelinks/v3.1/RouteLink_AK.nc"
    },
}

# process all routelinks
crosswalk_dict = {}
for domain in ['conus_31','hawaii','puertorico','alaska']:
    crosswalk_dict[domain] = extract_nwm_crosswalk_from_routelink(routelink_files[domain], SECONDARY_LOCATION_ID_PREFIX)

### Build crosswalk tables for each NWM version

In [ ]:
# v3.1 - conus_31, hawaii, puertorico, alaska
cross_list = []
for domain in ['conus_31','hawaii','puertorico','alaska']:
    cross_list.append(crosswalk_dict[domain])
nwm31_crosswalk = pd.concat(cross_list, ignore_index=True)
display(nwm31_crosswalk)

In [ ]:
UNFILTERED_CROSSWALK_DIR = Path("/data/common/crosswalks/routelinks/updated_crosswalks/")
nwm31_crosswalk.to_parquet(Path(UNFILTERED_CROSSWALK_DIR, "nwm31_crosswalk.parquet"), index=False)

### Filter out crosswalk primary_ids that are not in our updated locations table

In [ ]:
missing_df = pd.read_csv("/data/common/crosswalks/routelinks/missing_routelink_primary_ids_v3.1.csv")
print(missing_df.index.size)
missing_df.head()

In [ ]:
before_31 = nwm31_crosswalk.index.size
after_31 = nwm31_crosswalk[~nwm31_crosswalk["primary_location_id"].isin(missing_df["primary_location_id"])].index.size

In [ ]:
print(f"nwm31 before filter: {before_31}, after filter: {after_31}")

In [ ]:
nwm31_crosswalk = nwm31_crosswalk[~nwm31_crosswalk["primary_location_id"].isin(missing_df["primary_location_id"])]

### Write crosswalks to disk for loading 

In [ ]:
FILTERED_CROSSWALK_DIR = Path("/data/common/crosswalks/routelinks/updated_crosswalks_filtered/")

In [ ]:
nwm31_crosswalk.to_parquet(Path(FILTERED_CROSSWALK_DIR, "nwm31_crosswalk.parquet"), index=False)

### Load to TEEHR

In [ ]:
FILTERED_CROSSWALK_FILEPATH = Path(FILTERED_CROSSWALK_DIR, "nwm31_crosswalk.parquet")

In [ ]:
pd.read_parquet(FILTERED_CROSSWALK_FILEPATH)["properties"][0]

In [ ]:
from teehr.evaluation.evaluation import create_spark_session

spark = create_spark_session(
    aws_profile="admin-user"
)
ev = teehr.RemoteReadWriteEvaluation(spark=spark)

#### Make sure there are no primary IDs are in the filtered crosswalk that are not in our locations table

In [ ]:
sdf = ev.spark.read.parquet(FILTERED_CROSSWALK_FILEPATH.as_posix())
sdf.show(n=6, truncate=False)

In [ ]:
sdf.printSchema()

In [ ]:
sdf.count()

In [ ]:
xwalk_df = sdf.select("primary_location_id").distinct().toPandas()
xwalk_df.head()

In [ ]:
locations_df = ev.locations.to_pandas()

In [ ]:
mask = xwalk_df["primary_location_id"].isin(locations_df["id"])

In [ ]:
missing_xwalk_primary_ids_df = xwalk_df[~mask]  # this should be empty
missing_xwalk_primary_ids_df

In [ ]:
# missing_xwalk_primary_ids_df.to_csv("/data/common/crosswalks/routelinks/missing_routelink_primary_ids_v3.1.csv", index=False)

#### Load crosswalks to the warehouse

In [ ]:
%%time
ev.location_crosswalks.load_parquet(
    in_path=FILTERED_CROSSWALK_FILEPATH,
    write_mode="upsert"
)